In [1]:
!pip install datasets gensim rank_bm25 sentence-transformers numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 46.1 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
from gensim.models import Word2Vec
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import numpy as np
import time
import re

In [3]:
parallel = load_dataset("HSE-Chukchi-NLP/russian-chukchi-parallel-corpora")
df = parallel["train"]

doc_texts_ckt = list(df["ckt"])   # чукотские предложения — для индексов
doc_texts_ru  = list(df["ru"])    # русские переводы — только для отображения

# Фильтруем None (в датасете есть пропуски)
valid = [(ckt, ru) for ckt, ru in zip(doc_texts_ckt, doc_texts_ru)
         if ckt and ru and len(ckt.split(' ')) > 1]
doc_texts_ckt, doc_texts_ru = zip(*valid)
doc_texts_ckt = list(doc_texts_ckt)
doc_texts_ru  = list(doc_texts_ru)

print(f"Документов: {len(doc_texts_ckt)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


data.csv:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/70636 [00:00<?, ? examples/s]

Документов: 34078


In [4]:
def tokenize(text):
    text = text.lower()
    return re.findall(r'[\w\u0400-\u04FF\u0500-\u052F]+', text)

doc_tokens = [tokenize(text) for text in doc_texts_ckt]

In [5]:
def cosine_sim(v1, v2):
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    if denom == 0:
        return 0.0
    return np.dot(v1, v2) / denom

In [6]:
def print_results(results):
    for ckt, ru, score in results:
        print(f"[{score:.4f}]")
        print(f"  ckt: {ckt}")
        print(f"  ru:  {ru}")
        print()

### BM25

In [15]:
bm25_index = BM25Okapi(doc_tokens)

In [16]:
def search_bm25(query, top_k=5):
    query_tokens = tokenize(query)
    start = time.time()
    scores = bm25_index.get_scores(query_tokens)
    elapsed = time.time() - start
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    print(f"Время поиска BM25: {elapsed:.4f} сек\n")
    return [(doc_texts_ckt[i], doc_texts_ru[i], scores[i]) for i in top_indices]

## w2v

In [17]:
# Word2Vec
w2v_model = Word2Vec(
    sentences=doc_tokens,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    epochs=10
)


In [51]:
def text_to_vector(tokens, model):
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if not vecs:
        return np.zeros(model.vector_size)
    vec = np.mean(vecs, axis=0)
    norm = np.linalg.norm(vec)
    if norm == 0:
        return vec
    return vec / norm

doc_vectors_w2v = [text_to_vector(tokens, w2v_model) for tokens in doc_tokens]


In [48]:
def search_w2v(query, top_k=5):
    query_tokens = tokenize(query)
    query_vec = text_to_vector(query_tokens, w2v_model)
    start = time.time()
    scores = [cosine_sim(query_vec, dv) for dv in doc_vectors_w2v]
    elapsed = time.time() - start
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    print(f"Время поиска W2V: {elapsed:.4f} сек\n")
    return [(doc_texts_ckt[i], doc_texts_ru[i], scores[i]) for i in top_indices]


## FastText

In [38]:
from gensim.models import FastText

ft_model = FastText(
    sentences=doc_tokens,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    epochs=10
)

In [52]:
doc_vectors_ft  = [text_to_vector(tokens, ft_model)  for tokens in doc_tokens]

In [53]:
def search_ft(query, top_k=5):
    query_tokens = tokenize(query)
    query_vec = text_to_vector(query_tokens, ft_model)
    start = time.time()
    scores = [cosine_sim(query_vec, dv) for dv in doc_vectors_ft]
    elapsed = time.time() - start
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    print(f"Время поиска FastText: {elapsed:.4f} сек\n")
    return [(doc_texts_ckt[i], doc_texts_ru[i], scores[i]) for i in top_indices]

## Doc2Vec

In [59]:
from gensim.models import Doc2Vec
from gensim.models.doc2vec import TaggedDocument

tagged_docs = [TaggedDocument(tokens, [i]) for i, tokens in enumerate(doc_tokens)]

d2v_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    epochs=10
)

In [60]:
doc_vectors_d2v = [d2v_model.dv[i] for i in range(len(doc_tokens))]

In [61]:
doc_vectors_d2v = [v / (np.linalg.norm(v) + 1e-9) for v in doc_vectors_d2v]


In [62]:
def search_d2v(query, top_k=5):
    query_tokens = tokenize(query)
    # infer_vector для новых документов (запросов)
    query_vec = d2v_model.infer_vector(query_tokens, epochs=20)
    query_vec = query_vec / (np.linalg.norm(query_vec) + 1e-9)
    start = time.time()
    scores = [cosine_sim(query_vec, dv) for dv in doc_vectors_d2v]
    elapsed = time.time() - start
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    print(f"Время поиска Doc2Vec: {elapsed:.4f} сек\n")
    return [(doc_texts_ckt[i], doc_texts_ru[i], scores[i]) for i in top_indices]


## Random Indexing

In [67]:
import numpy as np
from collections import defaultdict

DIM = 1000       # размерность пространства
NONZERO = 6      # количество ненулевых элементов в signature

def make_signature():
    sig = np.zeros(DIM)
    indices = np.random.choice(DIM, NONZERO, replace=False)
    sig[indices[:NONZERO//2]] = 1
    sig[indices[NONZERO//2:]] = -1
    return sig

# Назначаем signature каждому документу
doc_signatures = [make_signature() for _ in range(len(doc_tokens))]

# Вектор слова = сумма signature документов где оно встречается
word_vectors = defaultdict(lambda: np.zeros(DIM))
for i, tokens in enumerate(doc_tokens):
    for token in set(tokens):  # set чтобы не суммировать дважды
        word_vectors[token] += doc_signatures[i]

# Вектор документа = среднее векторов его слов
def text_to_vector_ri(tokens):
    vecs = [word_vectors[t] for t in tokens if t in word_vectors]
    if not vecs:
        return np.zeros(DIM)
    vec = np.mean(vecs, axis=0)
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

doc_vectors_ri = [text_to_vector_ri(tokens) for tokens in doc_tokens]

def search_ri(query, top_k=5):
    query_vec = text_to_vector_ri(tokenize(query))
    start = time.time()
    scores = [cosine_sim(query_vec, dv) for dv in doc_vectors_ri]
    elapsed = time.time() - start
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    print(f"Время поиска RI: {elapsed:.4f} сек\n")
    return [(doc_texts_ckt[i], doc_texts_ru[i], scores[i]) for i in top_indices]

## search

In [69]:
def search(query, index="bm25", top_k=5):
    """
    query: строка запроса на чукотском
    index: "bm25" | "w2v" | "fasttext" | "doc2vec" | "ri"
    top_k: количество результатов
    """
    if index == "bm25":
        return search_bm25(query, top_k)
    elif index == "w2v":
        return search_w2v(query, top_k)
    elif index == "fasttext":
        return search_ft(query, top_k)
    elif index == "doc2vec":
        return search_d2v(query, top_k)
    elif index == "ri":
        return search_ri(query, top_k)
    else:
        raise ValueError(f"Неизвестный индекс: {index}. Выберите bm25, w2v, fasttext или doc2vec.")

In [54]:
results = search("ӄораӈы", index="bm25", top_k=5)
print_results(results)

Время поиска BM25: 0.0102 сек

[7.8210]
  ckt: очыткогъэ ӄораӈы.
  ru:  ответил олень.

[7.8210]
  ckt: Ӄораӈы татԓыӈыткой:
  ru:  Олень отвечает:

[7.8210]
  ckt: Ӄораӈы иквъи:
  ru:  Олень сказал:

[7.8210]
  ckt: Мынвэчгыргатын ӄораӈы
  ru:  Определим цену оленя

[7.8210]
  ckt: Ивыԓьыԓьын ӄораӈы
  ru:  Любящий мочу олень



In [55]:
results = search("ӄораӈы", index="w2v", top_k=5)
print_results(results)

Время поиска W2V: 0.2449 сек

[0.9997]
  ckt: Въиԓьын ӄораӈы
  ru:  Умерший олень

[0.9997]
  ckt: Ӄораӈы ныԓгинымкыӄин
  ru:  Оленей очень много

[0.9996]
  ckt: Аткэвма ӄэтвынин ӄораӈы
  ru:  Неумело он заколол оленя

[0.9996]
  ckt: Мынвэчгыргатын ӄораӈы
  ru:  Определим цену оленя

[0.9996]
  ckt: Гэԓеԓин ӄораӈы ӈирэнйъиԓгык
  ru:  Шёл олень два месяца



In [56]:
results = search("ӄораӈы", index="fasttext", top_k=5)
print_results(results)

Время поиска FastText: 0.2245 сек

[0.9999]
  ckt: Ынӄоры нэнанъомравморэ уннэԓгын гыттаԓянӈэԓга, нэнанкавравморэ ыннаныӈӄачагты.
  ru:  Далее закрепляем мембрану тонким жгутом, обматывая в один оборот.

[0.9999]
  ckt: Ынӄэн нъэԓык чавчыва энмэч тайкыёԓӄыԓ ӈэԓвыԓ, ыргынан гатата ӈэԓвыԓ ӄойвэгыргэты, миӈкы эгыӈ гатвата ывнкъам танрыԓгатык паӈъэвӈытонво.
  ru:  К этому времени и приурочивают пастухи выработку у телят полезных привычек, одновременно закрепляя их у всех оленей, делается это на пастбищах с хорошими местами отдыха для оленей - снежниками, речными наледями, куда специально подгоняют оленей.

[0.9999]
  ckt: Ӈинӄэй нэнмэйӈэвын Вантыргына ынкъам Энмынкауна.
  ru:  Воспитание мальчика взяли на себя Вантыргин и Энмынкау.

[0.9999]
  ckt: Гамыргогэчеԓен, таӈамчечьамыргота
  ru:  Он собрал морскую капусту, очень аппетитную морскую капусту

[0.9999]
  ckt: Ԓыгъоравэтԓьэн тэкэԓиӈыԓьын Юрий Рытгэв гэргыпатԓен ымы ӄоԓенотаеквэгты
  ru:  Чукотский писатель Юрий Рытхэу прославился на ве

In [66]:
results = search("Ивыԓьыԓьын ӄораӈы", index="doc2vec", top_k=5)
print_results(results)

Время поиска Doc2Vec: 0.6421 сек

[0.9780]
  ckt: Ынӄэнат Анфиса Шарыпова, Михаил Дьячков, Базик Добриев, Елена Евтюхова, Вера Грачева, Лидия Болина, Сергей Ыттъыкээв, Анатолий Тыӈэру, Алексей Тиркын, Евгения Тыӈэру, Елена Тэвлянкаав, Эльвира Венгер, Анна Отке, Людмила Данилова, Владимир Пуйъы, Любовь Вэрэгыргыӈа, Надежда Паӈарультыӈа, Елена Ойъыкэ, Лилия Эйӈэс, Валентина Кэвыԓӄут, Ивилина Кэԓьэк, Наталья Ԓиԓяԓио, Лариса Выквырагтыгыргыӈа, Андрей Носков, Тамара и Владимир Кликуновы, Людмила Лазутина, Дина Кэвӄэй, Светлана Кочнева, Александра Уяганская, Раиса Эйӈэԓӄут, Маргарита Меркулова ынкъам ӄутти.
  ru:  Это - Анфиса Шарыпова, Михаил Дьячков, Базик Добриев, Елена Евтюхова, Вера Грачёва, Лидия Болина, Сергей Эттыкеу, Анатолий Тынеру, Алексей Тиркин, Евгения Тынеру, Елена Тевлянкау, Эльвира Венгер, Анна Отке, Людмила Данилова, Владимир Пуя, Любовь Верегиргина, Надежда Панарультына, Елена Ойыке, Лилия Эйнес, Валентина Кеулькут, Ивилина Келек, Наталья Лилялио, Лариса Выквырагтыргыргына

In [71]:
results = search("ӄораӈы", index="ri", top_k=5)
print_results(results)

Время поиска RI: 0.5980 сек

[0.9950]
  ckt: Мынвэчгыргатын ӄораӈы
  ru:  Определим цену оленя

[0.9949]
  ckt: Ивыԓьыԓьын ӄораӈы
  ru:  Любящий мочу олень

[0.9949]
  ckt: Пыԓӄымԓяԓьын ӄораӈы
  ru:  Упитанный олень

[0.9851]
  ckt: Ӄораӈы татԓыӈыткой:
  ru:  Олень отвечает:

[0.9753]
  ckt: Ԓыгиӄымэк тыкынъун ӄораӈы
  ru:  Я чуть не поймал оленя

